# 第05課 - 能動RAG


## 設定

本筆記本展示了使用 Microsoft Agent Framework 的 Agentic RAG（檢索增強生成）模式。

**先決條件：**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — 你的 Azure AI 搜尋服務端點
- `AZURE_SEARCH_API_KEY` — 你的 Azure AI 搜尋 API 金鑰
- 透過環境變數配置的 Azure OpenAI 部署
- 已驗證 Azure CLI（`az login`）


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## 什麼是 Agentic RAG？

傳統的 RAG 採用固定流程：先檢索文件，然後生成回應。**Agentic RAG** 則進一步讓代理自主決定 <strong>何時</strong> 和 <strong>如何</strong> 進行資訊檢索。

使用 Agentic RAG，代理可以：
- <strong>決定</strong> 是否需要在回答問題前進行檢索
- <strong>選擇</strong> 要查詢的資料來源或工具
- <strong>評估</strong> 檢索到的結果，如果第一次嘗試不足，則進行後續檢索
- <strong>結合</strong> 多個檢索步驟中獲得的資訊，形成一致的答案

這使代理比固定的「先檢索後生成」流程更靈活且準確。


## 建立搜尋工具

在 Agentic RAG 中，外部資料來源會被包裝成代理人可以按需調用的<strong>工具</strong>。這讓代理人將檢索視為它可以採取的另一種動作，而不是必須執行的步驟。

以下我們定義了一個旅遊知識庫，並將其作為代理人可以呼叫的工具，來查詢目的地資訊。


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## 建立 RAG 智能代理

現在我們建立一個被指示 <strong>在回答前一定要先檢索資訊</strong> 的智能代理。該代理使用 `search_travel_knowledge` 工具，將回應以知識庫為依據，而非依賴其自身的訓練資料。


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## 反覆檢索 — 製作者-檢查者模式

Agentic RAG 的一大優勢是<strong>反覆檢索</strong>。代理人可以進行多輪搜尋，以驗證、修正或擴展其初步結果——類似於「製作者-檢查者」的工作流程：

1. <strong>製作者步驟</strong>：代理人檢索初步資訊並草擬回應。
2. <strong>檢查者步驟</strong>：代理人進行額外檢索以驗證細節或補足缺漏。

下方，代理人被問及一個需要比較多個目的地的問題，促使它多次搜尋。


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## 摘要

在本課程中，您學會了如何使用 Microsoft Agent Framework 建立一個 **Agentic RAG** 系統：

- **Agentic RAG** 讓代理能夠自主決定何時檢索資訊，使檢索過程動態而非固定。
- <strong>工具作為資料來源</strong>：外部知識庫（如 Azure AI 搜尋）被包裝成代理可調用的工具。
- <strong>迭代檢索</strong>：審核模式讓代理可以進行多輪檢索——搜尋、驗證與精煉——然後產生最終答案。

在生產環境中，您會用實際的 Azure AI 搜尋索引來取代記憶體中的 `TRAVEL_KNOWLEDGE_BASE`，以處理大規模旅遊文件檢索。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
此文件已使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們努力追求準確性，但請注意自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應視為權威來源。對於關鍵資訊，建議採用專業人工翻譯。我們不對因使用此翻譯所產生的任何誤解或誤譯承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
